In [0]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

In [0]:
spark = SparkSession.builder \
    .appName("Zomato Spark SQL Analysis") \
    .getOrCreate()

In [0]:
df_raw = spark.read.csv("/Volumes/workspace/default/zomato/zomato.csv", header=True, inferSchema=True)

In [0]:
clean_cols = [
    c.lower().strip().replace(" ", "_").replace("(", "").replace(")", "").replace("/", "_") 
    for c in df_raw.columns
]
df = df_raw.toDF(*clean_cols)

In [0]:
df_cleaned = df.withColumn("rating_value", F.regexp_extract(F.col("rate"), r"(\d+\.?\d*)", 1).cast("double")) \
               .withColumn("votes", F.col("votes").cast("integer")) \
               .withColumn("cost", F.regexp_replace(F.col("approx_costfor_two_people"), ",", "").cast("double")) \
               .withColumn("location_name", F.trim(F.col("location"))) \
               .withColumn("cuisine_name", F.trim(F.split(F.col("cuisines"), ",")[0])) \
               .withColumn("type", F.col("listed_intype"))

In [0]:
df_cleaned = df_cleaned.filter(
    F.col("name").isNotNull() & 
    F.col("location_name").isNotNull() & 
    F.col("cuisine_name").isNotNull()
)

In [0]:
df_cleaned = df_cleaned.dropDuplicates()


1.

In [0]:
locations_df = df_cleaned.select("location_name").distinct() \
    .withColumn("location_id", F.monotonically_increasing_id() + 1)


2.

In [0]:
cuisines_df = df_cleaned.select("cuisine_name").distinct() \
    .withColumn("cuisine_id", F.monotonically_increasing_id() + 1)

In [0]:
3.

4.

In [0]:
restaurants_df = df_cleaned \
    .join(locations_df, on="location_name", how="inner") \
    .join(cuisines_df, on="cuisine_name", how="inner") \
    .join(ratings_df, on=["rating_value", "votes"], how="inner") \
    .select(
        (F.monotonically_increasing_id() + 1).alias("restaurant_id"),
        F.col("name"),
        F.col("location_id"),
        F.col("cuisine_id"),
        F.col("rating_id"),
        F.col("cost"),
        F.col("type")
    ) \
    .dropDuplicates(["name", "location_id"])

5.

In [0]:
# Register all DataFrames as temporary SQL views
restaurants_df.createOrReplaceTempView("restaurants")
locations_df.createOrReplaceTempView("locations")
cuisines_df.createOrReplaceTempView("cuisines")
ratings_df.createOrReplaceTempView("ratings")
df.show()

In [0]:

# ==========================================
# PART C: Spark SQL Queries
# ==========================================

In [0]:
# Query 1: Retrieve restaurant names along with their location names
query_1 = spark.sql("""
    SELECT r.name AS restaurant_name, l.location_name
    FROM restaurants r
    JOIN locations l ON r.location_id = l.location_id
""")
df.show()

In [0]:
# Query 2: Find the top 10 restaurants with the highest ratings
query_2 = spark.sql("""
    SELECT r.name, rt.rating_value
    FROM restaurants r
    JOIN ratings rt ON r.rating_id = rt.rating_id
    ORDER BY rt.rating_value DESC
    LIMIT 10
""")
df.show()


In [0]:
# Query 3: Count the number of restaurants in each location
query_3 = spark.sql("""
    SELECT l.location_name, COUNT(r.restaurant_id) AS restaurant_count
    FROM locations l
    JOIN restaurants r ON l.location_id = r.location_id
    GROUP BY l.location_name
    ORDER BY restaurant_count DESC
""")
df.show()

In [0]:
# Query 4: Identify cuisines with the highest average rating
query_4 = spark.sql("""
    SELECT c.cuisine_name, ROUND(AVG(rt.rating_value), 2) AS avg_rating
    FROM restaurants r
    JOIN cuisines c ON r.cuisine_id = c.cuisine_id
    JOIN ratings rt ON r.rating_id = rt.rating_id
    GROUP BY c.cuisine_name
    ORDER BY avg_rating DESC
""")
df.show()

In [0]:
# Query 5: List restaurants with cost above the average cost
query_5 = spark.sql("""
    SELECT name, cost
    FROM restaurants
    WHERE cost > (SELECT AVG(cost) FROM restaurants)
""")
df.show()

In [0]:
# Query 6: Find the total number of votes received per location
query_6 = spark.sql("""
    SELECT l.location_name, SUM(rt.votes) AS total_votes
    FROM restaurants r
    JOIN locations l ON r.location_id = l.location_id
    JOIN ratings rt ON r.rating_id = rt.rating_id
    GROUP BY l.location_name
    ORDER BY total_votes DESC
""")
df.show()

In [0]:
# Query 7: Identify the most common cuisine in each location
query_7 = spark.sql("""
    WITH CuisineCounts AS (
        SELECT 
            l.location_name, 
            c.cuisine_name, 
            COUNT(r.restaurant_id) AS cuisine_count,
            ROW_NUMBER() OVER (
                PARTITION BY l.location_name 
                ORDER BY COUNT(r.restaurant_id) DESC
            ) as rank
        FROM restaurants r
        JOIN locations l ON r.location_id = l.location_id
        JOIN cuisines c ON r.cuisine_id = c.cuisine_id
        GROUP BY l.location_name, c.cuisine_name
    )
    SELECT location_name, cuisine_name, cuisine_count
    FROM CuisineCounts
    WHERE rank = 1
""")
df.show()


In [0]:
# Query 8: Retrieve restaurants that have ratings above 4 and votes greater than 500
query_8 = spark.sql("""
    SELECT r.name, rt.rating_value, rt.votes
    FROM restaurants r
    JOIN ratings rt ON r.rating_id = rt.rating_id
    WHERE rt.rating_value > 4.0 AND rt.votes > 500
""")
df.show()

In [0]:
# Query 9: Find the average cost for two people by cuisine
query_9 = spark.sql("""
    SELECT c.cuisine_name, ROUND(AVG(r.cost), 2) AS avg_cost_for_two
    FROM restaurants r
    JOIN cuisines c ON r.cuisine_id = c.cuisine_id
    GROUP BY c.cuisine_name
    ORDER BY avg_cost_for_two DESC
""")
df.show()

In [0]:
# Query 10: List locations where the number of restaurants exceeds a given threshold (e.g., threshold = 50)
threshold = 50
query_10 = spark.sql(f"""
    SELECT l.location_name, COUNT(r.restaurant_id) AS restaurant_count
    FROM locations l
    JOIN restaurants r ON l.location_id = r.location_id
    GROUP BY l.location_name
    HAVING COUNT(r.restaurant_id) > {threshold}
    ORDER BY restaurant_count DESC
""")
df.show()